# 04 — Feature extraction and dataset construction

**MATLAB equivalent:** `featsMat.m` and `extractEMGFeatures.m`

Each 40-sample × 16-channel window produces 48 features, in this exact order:

1. MAV for channels 1–16
2. RMS for channels 1–16
3. WL for channels 1–16

Rest receives label 0; movement labels are retained from `restimulus`.

In [ ]:
from pathlib import Path
import numpy as np
import joblib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "NinaPro_DB5"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SUBJECTS = 10
N_CHANNELS = 16
FS = 200
WINDOW_SAMPLES = 40
STEP_SAMPLES = 10
CLASS_ORDER = np.arange(13)  # 0 = rest, 1–12 = movements

split_data = joblib.load(OUTPUT_DIR / "Train_Test_DB.joblib")
TrainDB = split_data["TrainDB"]
TestDB = split_data["TestDB"]

## Feature function

The formulas and concatenation order match the MATLAB function.

In [ ]:
def extract_emg_features(window):
    """
    Parameters
    ----------
    window : ndarray, shape (samples, channels)

    Returns
    -------
    features : ndarray, shape (3 * channels,)
        [MAV channels..., RMS channels..., WL channels...]
    """
    window = np.asarray(window, dtype=np.float64)

    if window.ndim != 2:
        raise ValueError(f"Expected samples × channels, got {window.shape}")
    if window.shape[0] < 2:
        raise ValueError("A window must contain at least two samples.")

    mav = np.mean(np.abs(window), axis=0)
    rms = np.sqrt(np.mean(window ** 2, axis=0))
    wl = np.sum(np.abs(np.diff(window, axis=0)), axis=0)

    features = np.concatenate([mav, rms, wl])

    if not np.all(np.isfinite(features)):
        raise ValueError("Non-finite feature value encountered.")

    return features

def extract_window_features(windows):
    if len(windows) == 0:
        return np.empty((0, 3 * N_CHANNELS), dtype=np.float64)

    X = np.vstack([extract_emg_features(window) for window in windows])
    if X.shape[1] != 48:
        raise ValueError(f"Expected 48 features per window, got {X.shape[1]}")
    return X

## Construct training and testing matrices

The ordering follows the MATLAB script: for each subject, rest windows are appended before movement windows.

In [ ]:
def build_feature_dataset(database):
    X_parts, y_parts, subject_parts = [], [], []

    for s in range(1, N_SUBJECTS + 1):
        subject_name = f"S{s}"
        subject = database[subject_name]

        X_rest = extract_window_features(subject["Rest"]["EMG"])
        y_rest = np.zeros(len(X_rest), dtype=np.int64)

        X_movement = extract_window_features(subject["Movement"]["EMG"])
        y_movement = np.asarray(
            subject["Movement"]["MovementLabel"], dtype=np.int64
        )

        if len(X_movement) != len(y_movement):
            raise ValueError(f"Feature/label mismatch for {subject_name}")

        X_subject = np.vstack([X_rest, X_movement])
        y_subject = np.concatenate([y_rest, y_movement])
        subject_ids = np.full(len(y_subject), s, dtype=np.int64)

        X_parts.append(X_subject)
        y_parts.append(y_subject)
        subject_parts.append(subject_ids)

        print(f"{subject_name}: {len(y_subject):,} windows")

    if not X_parts:
        raise ValueError("No subject data found.")

    return (
        np.vstack(X_parts),
        np.concatenate(y_parts),
        np.concatenate(subject_parts),
    )

X_train, Y_train, TrainSubjectID = build_feature_dataset(TrainDB)
X_test, Y_test, TestSubjectID = build_feature_dataset(TestDB)

print("\\n===== FEATURE EXTRACTION COMPLETE =====")
print("Training matrix:", X_train.shape)
print("Testing matrix:", X_test.shape)
print("Training labels:", len(Y_train))
print("Testing labels:", len(Y_test))

## Dataset inspection

In [ ]:
import matplotlib.pyplot as plt

print("Unique training labels:", np.unique(Y_train))
print("Unique testing labels:", np.unique(Y_test))
print("NaN in training:", np.isnan(X_train).any())
print("Inf in training:", np.isinf(X_train).any())
print("NaN in testing:", np.isnan(X_test).any())
print("Inf in testing:", np.isinf(X_test).any())

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

axes[0].hist(Y_train, bins=np.arange(-0.5, 13.5, 1))
axes[0].set_title("Training Distribution")
axes[0].set_xlabel("Movement")
axes[0].set_ylabel("Windows")
axes[0].set_xticks(CLASS_ORDER)
axes[0].grid(True, alpha=0.3)

axes[1].hist(Y_test, bins=np.arange(-0.5, 13.5, 1))
axes[1].set_title("Testing Distribution")
axes[1].set_xlabel("Movement")
axes[1].set_xticks(CLASS_ORDER)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Save feature matrices

In [ ]:
feature_data = {
    "X_train": X_train,
    "Y_train": Y_train,
    "TrainSubjectID": TrainSubjectID,
    "X_test": X_test,
    "Y_test": Y_test,
    "TestSubjectID": TestSubjectID,
    "feature_order": ["MAV_ch1_to_16", "RMS_ch1_to_16", "WL_ch1_to_16"],
    "sampling_frequency_hz": FS,
    "window_samples": WINDOW_SAMPLES,
    "step_samples": STEP_SAMPLES,
}
joblib.dump(feature_data, OUTPUT_DIR / "feature_data.joblib", compress=3)
print("Saved:", OUTPUT_DIR / "feature_data.joblib")